# AI Project Control Tower — Conceptual Demo

This notebook illustrates the data model and workflow of the AI Project Control Tower using **static sample data**.

It does **not** call any live API endpoints and does **not** require a running Docker stack.
It uses only Python's standard library — no external dependencies.

For the full live experience, run `docker compose up --build` and open the Streamlit UI at http://localhost:8501.

---

## What this notebook covers

1. **Project and Blueprint structure** — what goes into a project record and a blueprint
2. **Finding model** — the structure of an audit finding with evidence, severity, and recommendation
3. **Audit dimensions** — the 9 quality dimensions every audit evaluates
4. **Scoring model** — how per-dimension scores are computed from findings
5. **Report section** — what a generated report section looks like
6. **System overview** — how the real pipeline connects these pieces

---

## 1. Project and Blueprint Structure

In [1]:
# Sample project record (mirrors the SQLAlchemy Project model in app/db/models/project.py)
project = {
    "id": 1,
    "name": "my-mlops-platform",
    "description": "Internal ML training and serving infrastructure",
    "repo_path": "/home/user/projects/my-mlops-platform",
    "created_at": "2026-05-02T10:00:00Z",
}

# Sample blueprint record (mirrors app/db/models/blueprint.py)
blueprint = {
    "id": 1,
    "project_id": 1,
    "name": "MLOps Platform Architecture v1",
    "content": """
        # Architecture Blueprint — MLOps Platform

        ## Required Components
        - FastAPI backend with async endpoints
        - PostgreSQL with pgvector for vector storage
        - Alembic for schema migrations
        - Prometheus metrics on /metrics endpoint
        - structlog for structured JSON logging
        - Docker Compose for local deployment
        - Unit + integration + E2E test suite
        - Streamlit UI for operator interface

        ## Security Requirements
        - No hardcoded credentials
        - Path allowlist for any file scanning
        - API keys loaded from environment variables only

        ## Documentation Requirements
        - README with architecture diagram
        - API reference documentation
        - Deployment guide
    """,
    "chunk_count": 12,
    "created_at": "2026-05-02T10:05:00Z",
}

print(f"Project: {project['name']}")
print(f"Blueprint: {blueprint['name']}")
print(f"Blueprint chunks: {blueprint['chunk_count']}")

Project: my-mlops-platform
Blueprint: MLOps Platform Architecture v1
Blueprint chunks: 12


---

## 2. Finding Model

Every audit finding has a consistent structure. In the real system, this is a Pydantic model
(`app/audit/models.py`) with typed fields and validation.

In [2]:
# Sample findings list (mirrors FindingModel from app/audit/models.py)
findings = [
    {
        "id": "f001",
        "audit_run_id": 1,
        "agent": "security",
        "dimension": "security",
        "severity": "high",
        "title": "No authentication layer on API endpoints",
        "file_path": "app/api/routes/audits.py",
        "evidence": "@router.post('/run') has no authentication dependency injected",
        "recommendation": "Add authentication middleware for production deployments. Document network isolation requirement for local-only use.",
    },
    {
        "id": "f002",
        "audit_run_id": 1,
        "agent": "devops_mlops",
        "dimension": "devops_mlops",
        "severity": "medium",
        "title": "Synchronous audit execution blocks the API process",
        "file_path": "app/audit/audit_engine.py",
        "evidence": "run_audit() is a blocking synchronous call with no background task queue",
        "recommendation": "Move long-running audits to a background worker queue for production workloads.",
    },
    {
        "id": "f003",
        "audit_run_id": 1,
        "agent": "rag_ai",
        "dimension": "rag_ai",
        "severity": "low",
        "title": "Hybrid RAG retrieval correctly implements RRF fusion",
        "file_path": "app/rag/hybrid_retriever.py",
        "evidence": "_rrf_fusion() combines TF-IDF and pgvector results with Reciprocal Rank Fusion",
        "recommendation": "Hybrid retrieval is correctly implemented per the Blueprint specification.",
    },
    {
        "id": "f004",
        "audit_run_id": 1,
        "agent": "documentation",
        "dimension": "documentation",
        "severity": "info",
        "title": "README includes architecture diagram and deployment guide",
        "file_path": "README.md",
        "evidence": "README.md contains Architecture section with ASCII component diagram and 'How to Run' section",
        "recommendation": "Documentation meets Blueprint requirements. Consider adding an ADR index.",
    },
    {
        "id": "f005",
        "audit_run_id": 1,
        "agent": "qa",
        "dimension": "testing_qa",
        "severity": "low",
        "title": "E2E non-modification test is present as a release gate",
        "file_path": "tests/e2e/test_no_repo_modification.py",
        "evidence": "test_no_repo_modification.py verifies target files are unchanged after audit run",
        "recommendation": "Release gate is correctly implemented. Ensure it runs in CI on every merge.",
    },
]

print(f"Total findings: {len(findings)}")
print()
for f in findings:
    print(f"[{f['severity'].upper():8}] {f['title']}")
    print(f"           File: {f['file_path']}")
    print()

Total findings: 5

[HIGH    ] No authentication layer on API endpoints
           File: app/api/routes/audits.py

[MEDIUM  ] Synchronous audit execution blocks the API process
           File: app/audit/audit_engine.py

[LOW     ] Hybrid RAG retrieval correctly implements RRF fusion
           File: app/rag/hybrid_retriever.py

[INFO    ] README includes architecture diagram and deployment guide
           File: README.md

[LOW     ] E2E non-modification test is present as a release gate
           File: tests/e2e/test_no_repo_modification.py



---

## 3. Audit Dimensions

Every audit evaluates the repository across 9 quality dimensions.
Each specialist agent is responsible for one or more dimensions.

In [3]:
# The 9 audit dimensions with agent ownership
DIMENSIONS = [
    {"key": "architecture",   "label": "Architecture",     "agent": "architecture"},
    {"key": "rag_ai",         "label": "RAG / AI",         "agent": "rag_ai"},
    {"key": "devops_mlops",   "label": "DevOps / MLOps",   "agent": "devops_mlops"},
    {"key": "testing_qa",     "label": "Testing / QA",     "agent": "qa"},
    {"key": "security",       "label": "Security",         "agent": "security"},
    {"key": "documentation",  "label": "Documentation",    "agent": "documentation"},
    {"key": "observability",  "label": "Observability",    "agent": "architecture"},
    {"key": "data_management","label": "Data Management",  "agent": "devops_mlops"},
    {"key": "overall",        "label": "Overall",          "agent": "orchestrator"},
]

# Severity weights used by the scoring engine
SEVERITY_WEIGHTS = {
    "critical": -30,
    "high":     -15,
    "medium":   -8,
    "low":      -3,
    "info":      0,
}

print("Audit Dimensions:")
print("-" * 50)
for d in DIMENSIONS:
    print(f"  {d['label']:20} → agent: {d['agent']}")

Audit Dimensions:
--------------------------------------------------
  Architecture         → agent: architecture
  RAG / AI             → agent: rag_ai
  DevOps / MLOps       → agent: devops_mlops
  Testing / QA         → agent: qa
  Security             → agent: security
  Documentation        → agent: documentation
  Observability        → agent: architecture
  Data Management      → agent: devops_mlops
  Overall              → agent: orchestrator


---

## 4. Scoring Model

Scores start at 100 and are reduced by the severity weights of findings in each dimension.
The overall score is the mean of all dimension scores.

In [4]:
def compute_scores(findings, dimensions, severity_weights):
    """Compute per-dimension scores from a list of findings."""
    dim_scores = {}

    for d in dimensions:
        if d["key"] == "overall":
            continue
        # Start at 100, subtract penalty for each finding in this dimension
        score = 100
        dim_findings = [f for f in findings if f["dimension"] == d["key"]]
        for finding in dim_findings:
            score += severity_weights.get(finding["severity"], 0)
        dim_scores[d["key"]] = max(0, min(100, score))  # Clamp to [0, 100]

    # Overall = mean of all dimension scores
    if dim_scores:
        dim_scores["overall"] = round(sum(dim_scores.values()) / len(dim_scores))

    return dim_scores


scores = compute_scores(findings, DIMENSIONS, SEVERITY_WEIGHTS)

print("Audit Scores:")
print("-" * 40)
for dim in DIMENSIONS:
    key = dim["key"]
    score = scores.get(key, "n/a")
    if isinstance(score, int):
        bar = "█" * (score // 10) + "░" * (10 - score // 10)
        print(f"  {dim['label']:20} {bar} {score:3}")
    else:
        print(f"  {dim['label']:20} {score}")

Audit Scores:
----------------------------------------
  Architecture         ██████████ 100
  RAG / AI             █████████░  97
  DevOps / MLOps       █████████░  92
  Testing / QA         █████████░  97
  Security             ████████░░  85
  Documentation        ██████████ 100
  Observability        ██████████ 100
  Data Management      ██████████ 100
  Overall              █████████░  96


---

## 5. Sample Report Section

The report generator produces Markdown, HTML, and JSON. This cell shows what a Markdown
section looks like when rendered from the findings list.

In [5]:
def generate_markdown_report(project, blueprint, findings, scores):
    """Generate a simple Markdown report from project metadata and findings."""
    severity_order = {"critical": 0, "high": 1, "medium": 2, "low": 3, "info": 4}
    sorted_findings = sorted(findings, key=lambda f: severity_order.get(f["severity"], 99))

    lines = [
        f"# Audit Report — {project['name']}",
        f"",
        f"**Project:** {project['name']}",
        f"**Blueprint:** {blueprint['name']}",
        f"**Overall Score:** {scores.get('overall', 'n/a')} / 100",
        f"",
        "---",
        "",
        "## Scores",
        "",
        "| Dimension | Score |",
        "|---|---|",
    ]
    for key, score in scores.items():
        lines.append(f"| {key.replace('_', ' ').title()} | {score} |")

    lines += ["", "---", "", "## Findings", ""]

    for f in sorted_findings:
        lines += [
            f"### [{f['severity'].upper()}] {f['title']}",
            f"",
            f"- **Agent:** {f['agent']}",
            f"- **Dimension:** {f['dimension']}",
            f"- **File:** `{f['file_path']}`",
            f"",
            f"**Evidence:**",
            f"",
            f"```",
            f"{f['evidence']}",
            f"```",
            f"",
            f"**Recommendation:** {f['recommendation']}",
            f"",
            "---",
            "",
        ]

    return "\n".join(lines)


report_md = generate_markdown_report(project, blueprint, findings, scores)
print(report_md[:2000])  # Print first 2000 chars for preview

# Audit Report — my-mlops-platform

**Project:** my-mlops-platform
**Blueprint:** MLOps Platform Architecture v1
**Overall Score:** 96 / 100

---

## Scores

| Dimension | Score |
|---|---|
| Architecture | 100 |
| Rag Ai | 97 |
| Devops Mlops | 92 |
| Testing Qa | 97 |
| Security | 85 |
| Documentation | 100 |
| Observability | 100 |
| Data Management | 100 |
| Overall | 96 |

---

## Findings

### [HIGH] No authentication layer on API endpoints

- **Agent:** security
- **Dimension:** security
- **File:** `app/api/routes/audits.py`

**Evidence:**

```
@router.post('/run') has no authentication dependency injected
```

**Recommendation:** Add authentication middleware for production deployments. Document network isolation requirement for local-only use.

---

### [MEDIUM] Synchronous audit execution blocks the API process

- **Agent:** devops_mlops
- **Dimension:** devops_mlops
- **File:** `app/audit/audit_engine.py`

**Evidence:**

```
run_audit() is a blocking synchronous call with n

---

## 6. How the Real System Works

In the live application, these steps are handled by the following components:

| Step | Component | Location |
|---|---|---|
| Path validation | `PathValidator` | `app/scanner/path_validator.py` |
| File scanning | `RepoScanner` | `app/scanner/repo_scanner.py` |
| Secret masking | `SecretMasker` | `app/scanner/secret_masker.py` |
| Blueprint chunking | `Chunker` | `app/rag/chunker.py` |
| RAG indexing | `RAGIndexer` | `app/rag/indexer.py` |
| Hybrid retrieval | `HybridRetriever` | `app/rag/hybrid_retriever.py` |
| Agent orchestration | `OrchestratorAgent` | `app/agents/orchestrator_agent.py` |
| Scoring | `ScoringEngine` | `app/audit/scoring.py` |
| Report generation | `ReportGenerator` | `app/reports/report_generator.py` |
| Report sanitisation | `ReportSanitiser` | `app/reports/report_sanitizer.py` |

### To run the real system:

```bash
cp .env.example .env
docker compose up --build
docker compose exec api alembic upgrade head
# Open http://localhost:8501
```

### Key invariant:

The system **never modifies the target repository**. This is enforced at multiple layers
and verified by a mandatory E2E release gate:

```bash
pytest tests/e2e/test_no_repo_modification.py -v
```